# Route profile

Цель первого шага: создать признаки маршрута по точному `Flight Distance` и проверить их на `fold 0`.

Сначала добавим только три понятных признака на примере `Online boarding`. После проверки обобщим код на остальные оценки.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_data_dir = project_root / "data" / "processed"

train_df = pd.read_parquet(processed_data_dir / "train_with_folds.parquet")
test_df = pd.read_parquet(processed_data_dir / "test_clean.parquet")

print("train:", train_df.shape)
print("test:", test_df.shape)

train: (699635, 25)
test: (299844, 23)


## Разделяем один фолд

Профиль маршрута считаем только по `x_train`. Данные из `x_valid` не участвуют в расчёте средних — иначе возникнет утечка.

In [3]:
valid_fold = 0
target = "satisfaction"
drop_columns = ["id", target, "fold"]

train_mask = train_df["fold"] != valid_fold
valid_mask = train_df["fold"] == valid_fold

x_train = train_df.loc[train_mask].drop(columns=drop_columns).copy()
x_valid = train_df.loc[valid_mask].drop(columns=drop_columns).copy()
x_test = test_df.drop(columns=["id"]).copy()

y_train = train_df.loc[train_mask, target]
y_valid = train_df.loc[valid_mask, target]

print("x_train:", x_train.shape)
print("x_valid:", x_valid.shape)
print("x_test:", x_test.shape)

x_train: (559708, 22)
x_valid: (139927, 22)
x_test: (299844, 22)


## Первые признаки маршрута

Для каждого значения `Flight Distance` считаем по обучающей части:

- `route_count` — сколько пассажиров было на маршруте;
- `route_online_boarding_mean` — средняя оценка онлайн-регистрации;
- `route_online_boarding_residual` — насколько оценка текущего пассажира отличается от средней по маршруту.

In [4]:
route_column = "Flight Distance"
profile_column = "Online boarding"

route_counts = x_train[route_column].value_counts()
route_means = x_train.groupby(route_column)[profile_column].mean()
global_mean = x_train[profile_column].mean()

for frame in [x_train, x_valid, x_test]:
    frame["route_count"] = (
        frame[route_column].map(route_counts).fillna(0).astype("float32")
    )

    frame["route_online_boarding_mean"] = (
        frame[route_column].map(route_means).fillna(global_mean).astype("float32")
    )

    frame["route_online_boarding_residual"] = (
        frame[profile_column] - frame["route_online_boarding_mean"]
    ).astype("float32")

In [5]:
route_features = [
    "route_count",
    "route_online_boarding_mean",
    "route_online_boarding_residual",
]

print("new shapes:", x_train.shape, x_valid.shape, x_test.shape)
print("missing values:")
print(x_valid[route_features].isna().sum())

x_valid[[route_column, profile_column] + route_features].head()

new shapes: (559708, 25) (139927, 25) (299844, 25)
missing values:
route_count                       0
route_online_boarding_mean        0
route_online_boarding_residual    0
dtype: int64


,Flight Distance,Online boarding,route_count,route_online_boarding_mean,route_online_boarding_residual
2,1371,3,309.0,3.330097,-0.330097
9,588,3,3142.0,3.098027,-0.098027
11,2636,2,12.0,3.583333,-1.583333
13,946,2,2535.0,3.035897,-1.035897
17,631,2,2371.0,2.901729,-0.901729
